In [ ]:
%pip install numpy
%pip install imageio
%pip install pygraphviz
%pip install matplotlib
%pip install networkx
%pip install tqdm
%pip install pandas

## Problema das N-rainhas

### Visão geral do problema

O problema das n-rainhas consiste em posicionar n rainhas em um tabuleiro de tamanho n de modo que nenhuma rainha ameace a outra.

O caso das 8 rainhas em um tabuleiro de tamanho 8x8 é um caso especial do problema mais geral, existindo 92 soluções possíveis.

Apesar de classicamente se buscar por todas as soluções possíveis para cada conjunto de n rainhas e tabuleiro n x n, na implementação abaixo o objetivo é encontrar a primeira solução e exibir, de modo que seja possível avaliar qual o melhor algoritmo de busca a ser utilizado para encontrar a solução de maneira mais rápida. Cabe destacar ainda que, os algoritmos utilizados foram apenas o de busca em largura (Breadth First Search - BFS) e o de profundidade (Depth First Search - DFS), para n's = 4, 5, 6 e 8. Foi tentado utilizar n = 16, porém houve estouro de memória para a BFS, se tornando inviável encontrar uma solução para esse tipo de busca, pois, apesar do problema das n-rainhas gerar uma árvore considerada profunda, a árvore tem uma característica peculiar, dado que seu fator de ramificação começa em N e diminui a cada nível, tornando o topo da árvore **muito largo**, mas a base estreita. Nesse sentido, como a BFS usa uma FIFO a largura da árvore no topo aumenta consideravelmente para n's >= 16, mesmo que a quantidade de nós-filhos a cada nível seja menor do que a quantidade de nós do nível superior, por naturalmente já ser larga se torna praticamente impossivel de encontrar uma solução utilizando uma máquina com pouca memória e, mais ainda, utilizando código em python de forma não otimizada para o tamanho do problema abordado.

### Informações do computador

* Processadores: 8 × AMD Ryzen 7 7800X3D 8-Core Processor
* Memória: 32 GiB de RAM (30,5 GiB utilizável)

In [40]:
from collections import deque
import time
import pandas as pd
import pygraphviz
import imageio
import numpy as np 
import matplotlib.pyplot as plt
import networkx as nx
from tqdm import tqdm

In [41]:
class QueensBoardChess:

    def __init__(self, size_board: int):
        self.n = size_board
        self.board = np.zeros((self.n, self.n))
    
    def position_queen(self, row: int, column: int) -> bool:
        if 0 <= row < self.n and 0 <= column < self.n:
            self.board[row][column] = 1
            return True
        else:
            return False
    
    def remove_queen(self, row: int, column: int) -> bool:
        if 0 <= row < self.n and 0 <= column < self.n:
            self.board[row][column] = 0
            return True
        else:
            return False
    
    def valid_position(self, row: int, column: int) -> bool:
        for i in range(row):
            if self.board[i][column] == 1:
                return False

        i, j = row, column
        while i >= 0 and j >= 0:
            if self.board[i][j] == 1:
                return False
            i -= 1
            j -= 1

        k, y = row, column
        while k >= 0 and y < self.n:
            if self.board[k][y] == 1:
                return False
            k -= 1
            y += 1

        return True

In [42]:
class Problem:

    def __init__(self, initial_state, actions, transition_model, goal_test, step_cost):

        self.initial_state = initial_state
        self.actions = actions                    
        self.transition_model = transition_model  
        self.goal_test = goal_test                
        self.step_cost = step_cost  

class Node:

    def __init__(self, problem, parent = None, action = None):

        self.parent = parent
        self.action = action

        if parent is None:
            self.state = problem.initial_state
            self.path_cost = 0.0
        else:
            self.state = problem.transition_model(parent.state, action)
            self.path_cost = parent.path_cost + problem.step_cost(parent.state, action)


    def __eq__(self, other):

        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):

        return hash(self.state)

    def solution(self):

        node = self
        path = []

        while node.parent is not None:
            path.append(node.action)
            node = node.parent
        path.reverse()
        
        return path

In [43]:
class QueensProblem(Problem):

    def __init__(self, start_node, size_board: int):

        self.board_chess = QueensBoardChess(size_board)

        super().__init__(initial_state = start_node,
                         actions = self.actions_fn,
                         transition_model = self.transition_fn,
                         goal_test = self.goal_test_fn,
                         step_cost = self.step_cost_fn)

    def actions_fn(self, state):

        actions = []
        current_row = len(state)

        if current_row >= self.board_chess.n:
            return actions

        self.board_chess.board.fill(0)
        for each_row, each_column in enumerate(state):
            self.board_chess.position_queen(each_row, each_column)

        for column in range(self.board_chess.n):
            if self.board_chess.valid_position(current_row, column):
                actions.append((current_row, column))  
        return actions

    def transition_fn(self, state, action):

        row, col = action
        return state + (col,)

    def goal_test_fn(self, state):
            
        return len(state) == self.board_chess.n

    def step_cost_fn(self, state, action, uniform_cost = True):
        return 1

In [44]:
def state_to_board_str(state, n):
    grid = [["." for _ in range(n)] for _ in range(n)]
    for r, c in enumerate(state):
        grid[r][c] = "Q"
    return "\n".join(" ".join(row) for row in grid)

In [53]:
class BreadthFirstSearch:

    def __init__(self, problem):

        self.problem = problem
        self.frontier = deque()  
        self.explored = set()    
        self.tree_graph = nx.DiGraph()
        self.history = []

    def search(self):

        start_time = time.perf_counter()
        root = Node(problem = self.problem)
        n = self.problem.board_chess.n

        board_label = state_to_board_str(root.state, n)
        self.tree_graph.add_node(id(root), state = str(root.state), label = board_label)

        self.frontier.append(root)

        solution_node = None
        node_count = 0

        with tqdm(desc = "Executando BFS", unit = "º nó") as pbar:
            while self.frontier:
                node = self.frontier.popleft()
                node_count += 1
                self.explored.add(node)

                active_nodes = list(self.tree_graph.nodes())
                self.history.append((active_nodes, id(node)))

                if self.problem.goal_test(node.state):
                    solution_node = node
                    pbar.update(1)
                    break

                for action in self.problem.actions(node.state):

                    child = Node(self.problem, node, action)
                    child_label = state_to_board_str(child.state, n)
                    self.tree_graph.add_node(id(child), state = str(child.state), label=child_label)
                    self.tree_graph.add_edge(id(node), id(child))

                    if (child not in self.explored) and (child not in self.frontier):
                        self.frontier.append(child)

                pbar.set_postfix(Explored = f": {len(self.explored)}", Frontier = f": {len(self.frontier)}")
                pbar.update(1)

        end_time = time.perf_counter()

        return {
            "Algoritmo": "BFS",
            "N (Board)": n,
            "Solução": solution_node.state if solution_node else None,
            "Nós explorados/nó da solução": len(self.explored),
            "Nós na Fronteira": len(self.frontier),
            "Tempo (s)": round(end_time - start_time, 4)
        }

In [54]:
class DepthFirstSearch:

    def __init__(self, problem):
        self.problem = problem
        self.frontier = []
        self.explored = set()
        self.tree_graph = nx.DiGraph()
        self.history = []

    def search(self):
        start_time = time.perf_counter()
        root = Node(problem = self.problem)
        n = self.problem.board_chess.n

        board_label = state_to_board_str(root.state, n)
        self.tree_graph.add_node(id(root), state = str(root.state), label = board_label)

        self.frontier.append(root)
        solution_node = None
        node_count = 0

        with tqdm(desc = "Executando DFS", unit = "º nó") as pbar:
            while self.frontier:
                node = self.frontier.pop()

                if node in self.explored:
                    continue

                node_count += 1
                self.explored.add(node)

                active_nodes = list(self.tree_graph.nodes())
                self.history.append((active_nodes, id(node)))

                if self.problem.goal_test(node.state):
                    solution_node = node
                    pbar.update(1)
                    break

                for action in reversed(list(self.problem.actions(node.state))):

                    child = Node(self.problem, node, action)
                    child_label = state_to_board_str(child.state, n)
                    self.tree_graph.add_node(id(child), state = str(child.state), label = child_label)
                    self.tree_graph.add_edge(id(node), id(child))

                    if (child not in self.explored) and (child not in self.frontier):
                        self.frontier.append(child)

                pbar.set_postfix(Explored = f" {len(self.explored)}", Frontier = f" {len(self.frontier)}")
                pbar.update(1)

        end_time = time.perf_counter()

        return {
            "Algoritmo": "DFS",
            "N (Board)": n,
            "Solução": solution_node.state if solution_node else None,
            "Nós explorados/nó da solução": len(self.explored),
            "Nós na Fronteira": len(self.frontier),
            "Tempo (s)": round(end_time - start_time, 4)
        }

In [47]:
def generate_animation(graph, steps, filename = "n_queens_search.gif"):
    frames = []

    pos = nx.drawing.nx_agraph.graphviz_layout(graph, prog = "dot")
    fig, ax = plt.subplots(figsize = (22, 10))

    for step, (active_nodes, current_node) in tqdm(list(enumerate(steps))):
        ax.clear()
        subgraph = graph.subgraph(active_nodes)

        node_colors = [
            "lightgreen" if n == current_node else "white" 
            for n in subgraph.nodes()
        ]

        labels = {n: graph.nodes[n]["label"] for n in subgraph.nodes()}

        nx.draw(
            subgraph,
            pos,
            labels = labels,
            node_color = node_colors,
            node_shape = "s",        
            node_size = 3500,      
            font_size = 8,
            font_family = "monospace", 
            edgecolors = "black",   
            arrowsize = 12,
            ax = ax,
        )
        ax.set_title(f"Passo {step + 1}: Expansão da Árvore de Busca", fontsize = 12)

        fig.canvas.draw()
        image = imageio.core.util.Array(
            np.array(fig.canvas.buffer_rgba())
        )

        frames.append(image)

    plt.close()
    
    imageio.mimsave(filename, frames, fps = 1) 



In [55]:
problem_instance = QueensProblem(start_node = (), size_board = 4)

print("Breadth-First Search (BFS):")

bfs = BreadthFirstSearch(problem_instance)
result1 = bfs.search()

if result1:
    print("Solução encontrada (colunas por linha):", result1["Solução"])
print("Nós explorados (BFS):", len(bfs.explored))
generate_animation(bfs.tree_graph, bfs.history, filename="n_queens_bfs.gif")

print("\nDepth-First Search (DFS):")

dfs = DepthFirstSearch(problem_instance)
result2 = dfs.search()

if result2:
    print("Solução DFS (colunas por linha):", result2["Solução"])
print("Nós explorados (DFS):", len(dfs.explored))
generate_animation(dfs.tree_graph, dfs.history, filename="n_queens_dfs.gif")

Breadth-First Search (BFS):


Executando BFS: 16º nó [00:00, 3624.57º nó/s, Explored=: 15, Frontier=: 2]


Solução encontrada (colunas por linha): (1, 3, 0, 2)
Nós explorados (BFS): 16


100%|██████████| 16/16 [00:00<00:00, 20.31it/s]



Depth-First Search (DFS):


Executando DFS: 9º nó [00:00, 5768.45º nó/s, Explored=8, Frontier=3]


Solução DFS (colunas por linha): (1, 3, 0, 2)
Nós explorados (DFS): 9


100%|██████████| 9/9 [00:00<00:00, 27.72it/s]


In [ ]:
board_sizes = [4, 5, 6, 8, 16]
results = []

for size in board_sizes:
    problem_bfs = QueensProblem(start_node = (), size_board = size)
    bfs = BreadthFirstSearch(problem_bfs)
    res_bfs = bfs.search()
    results.append(res_bfs)

    problem_dfs = QueensProblem(start_node = (), size_board = size)
    dfs = DepthFirstSearch(problem_dfs)
    res_dfs = dfs.search()
    results.append(res_dfs)

df = pd.DataFrame(results)

print()
print(df.to_string(index = False))

nome_arquivo = "resultado_n_rainhas.csv"
df.to_csv(nome_arquivo, index = False, sep = ";", encoding = "utf-8-sig")

print(f"\nTabela salva com sucesso no arquivo: {nome_arquivo}")

Executando BFS: 16º nó [00:00, 4349.81º nó/s, Explored=: 15, Frontier=: 2]
Executando DFS: 9º nó [00:00, 6922.56º nó/s, Explored=8, Frontier=3]
Executando BFS: 45º nó [00:00, 4405.78º nó/s, Explored=: 44, Frontier=: 10]
Executando DFS: 6º nó [00:00, 5194.18º nó/s, Explored=5, Frontier=7]
Executando BFS: 150º nó [00:00, 4208.98º nó/s, Explored=: 149, Frontier=: 4] 
Executando DFS: 32º nó [00:00, 3939.93º nó/s, Explored=31, Frontier=9]
Executando BFS: 1966º nó [00:00, 3389.99º nó/s, Explored=: 1965, Frontier=: 92] 
Executando DFS: 114º nó [00:00, 4840.12º nó/s, Explored=113, Frontier=12]
Executando BFS: 34888º nó [31:28,  3.77º nó/s, Explored=: 34888, Frontier=: 204050]